# **Practical AI (ST1508) CA 1**

In this Jupyter Notebook, the sensor dataset that have been provided will be prepared and cleaned. This will prepare the way for Data Visualisations which will be used to obtain insights. Subsequently, the data will be inserted into the SQL Data Warehouse and analysed. We will later proceed to conduct the Model Training using the Data from the Data Warehouse

---
# 1. Libraries Import and Environment Setup

Before conducting Data Preparation and Cleaning, we will firstly import the required modules and libraries. These modules and libraries are required to perform the Data Cleaning with the datasets. Subsequently, we will setup the environment so as to ensure the ETL Pipeline is consistent with every batch of data operations. This also makes the ETL Pipeline replicable.

In [ ]:
# 1) Install MySQL Python client
!pip install -q mysql-connector-python

import gc
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType
)

import os

from IPython.display import display

import pandas as pd
from typing import List, Dict
from IPython.display import display
import mysql.connector

from pyspark.sql.window import Window
from pyspark.sql import functions as F

from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA

from pyspark.sql import types as T
import numpy as np
from IPython.display import display

from pyspark.ml.feature import Imputer

In [ ]:
# === S0: Setup — install MySQL connector + start Spark (Colab) ==============
"""
Run this ONCE per Colab runtime.

- Installs mysql-connector-python for talking to MySQL/RDS.
- Starts a simple SparkSession (no JDBC jars needed).
"""

# 2) Kill any old Spark session just in case
try:
    spark.stop()
except Exception:
    pass

gc.collect()

# 3) Start fresh Spark (Colab already has PySpark installed)
spark = (
    SparkSession.builder
    .appName("PAI_CA1_Sensor_Cleaning")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .getOrCreate()
)

print("Spark started.")
spark.range(5).show()


With reference to the Code Cell above, we are able to determine the required libraries have been successfully imported and we currently have them in the local environment. We are also able to determine that the Google Drive have been mounted, implying we are able to access the provided dataset in the local environment.

---
# 2. Data Extract Layer

In this section, we will be conducting the Extraction component of the ETL Process. The Extraction process will be defined as conducting the data retrieval from the Staging Database and conducting quality checks on the Data without performing any operations yet. This step is crucial as it will allow us to start the pipeline with the latest data and allow for the cleaning of the data.

---
## 2.1 Data Extraction From Staging Database

In this sub-section, we will be extracting the Data from the Staging Database. This process is known as the Extract segment within the ETL pipeline. The data will be queried using SQL from the mySQL database and subsequently it will go through the Transform Layer after the Extraction is done successfully.

In [ ]:
# === S1: MySQL → pandas (typed) → Spark df_raw ==============================
"""
Loads raw sensor_data_source rows from AWS RDS (where cleaned_date IS NULL),
fixes dtypes in pandas, converts to a strongly-typed Spark DataFrame df_raw,
then drops the pandas DataFrame to free RAM.

Final schema (df_raw):
    csv_index      : string
    booking_id     : string
    accuracy       : double
    bearing        : double
    acceleration_x : double
    acceleration_y : double
    acceleration_z : double
    gyro_x         : double
    gyro_y         : double
    gyro_z         : double
    second         : double
    speed          : double
    cleaned_date   : string
"""

# ---------- MySQL / RDS connection config ----------
db_user  = "admin"
db_pass  = os.getenv("STAGING_DB_PASSWORD")
db_host  = os.getenv("STAGING_DB_HOST")
db_port  = 3306
db_name  = "staging"
db_table = "sensor_data_source"

# ---------- SQL query ----------
SELECT_COLS = [
    "csv_index",
    "booking_id",
    "accuracy",
    "bearing",
    "acceleration_x", "acceleration_y", "acceleration_z",
    "gyro_x", "gyro_y", "gyro_z",
    "`second`", "speed",
    "cleaned_date",
]
select_list = ", ".join(SELECT_COLS)

query = f"""
SELECT
    {select_list}
FROM {db_table}
WHERE cleaned_date IS NULL
"""

print("Running SQL query against MySQL:\n", query)

# ---------- Connect & load to pandas ----------
cnx = mysql.connector.connect(
    host=db_host,
    port=db_port,
    user=db_user,
    password=db_pass,
    database=db_name,
)

pdf = pd.read_sql(query, cnx)
cnx.close()

print("\n[MySQL → pandas] Shape:", pdf.shape)
print(pdf.head())
print("\n[MySQL → pandas] dtypes BEFORE fix:")
print(pdf.dtypes)

# ---------- Fix pandas dtypes explicitly ----------
NUMERIC_COLS = [
    "accuracy", "bearing",
    "acceleration_x", "acceleration_y", "acceleration_z",
    "gyro_x", "gyro_y", "gyro_z",
    "second", "speed",
]

# Make sure id-like columns stay as strings
pdf["csv_index"]  = pdf["csv_index"].astype(str)
pdf["booking_id"] = pdf["booking_id"].astype(str)

for c in NUMERIC_COLS:
    pdf[c] = pd.to_numeric(pdf[c], errors="coerce")

# cleaned_date as string (even if all None for now)
pdf["cleaned_date"] = pdf["cleaned_date"].astype(str)

print("\n[MySQL → pandas] dtypes AFTER fix:")
print(pdf.dtypes)

# ---------- Define explicit Spark schema ----------
schema = StructType([
    StructField("csv_index",      StringType(), True),
    StructField("booking_id",     StringType(), True),
    StructField("accuracy",       DoubleType(), True),
    StructField("bearing",        DoubleType(), True),
    StructField("acceleration_x", DoubleType(), True),
    StructField("acceleration_y", DoubleType(), True),
    StructField("acceleration_z", DoubleType(), True),
    StructField("gyro_x",         DoubleType(), True),
    StructField("gyro_y",         DoubleType(), True),
    StructField("gyro_z",         DoubleType(), True),
    StructField("second",         DoubleType(), True),
    StructField("speed",          DoubleType(), True),
    StructField("cleaned_date",   StringType(), True),
])

# ---------- Convert pandas → Spark with schema ----------
df_raw = spark.createDataFrame(pdf, schema=schema)

print("\n[Spark] df_raw schema:")
df_raw.printSchema()

print("\n[Spark] Sample rows from df_raw:")
df_raw.show(5, truncate=False)

# ---------- Clean up pandas to free RAM ----------
del pdf
gc.collect()

print("\n[CLEANUP] Dropped pandas DataFrame; df_raw is now the source of truth.")

With reference to the Code Cell above, we are able to determine that the Extraction of the Data from the Staging Database have been successfully completed and we are able to proceed to execute the Data Quality Check operations in the next sub-section.

---
## 2.2 Data Quality Inspection

In this sub-section, we will be conducting Data Quality Inspection on the Sensor Data extracted from the Staging Database. This will allow us to determine what is required for the pipeline and what we can expect to encounter when dealing with Data Abnormalities. This process is under the Extraction Phase of the ETL Process as there are no physical changes done to the Data except for extracting the Data Quality Report. The inspection will be conducted in the Code Cell below.

In [ ]:
"""
Cell S2 — Baseline Audit (Spark, full df_raw)

Uses df_raw from S1 (no extra MySQL hits).

What this cell does:
  1) Uses the FULL df_raw as the audit frame.
  2) Computes missing values per column.
  3) Captures a data types snapshot from Spark.
  4) Computes duplicate summary (full-row equality).
"""

print("Building audit view from FULL df_raw (no sampling) ...")

# Use full df_raw for audit
df_audit = df_raw.cache()
total_rows = df_audit.count()
cols = df_audit.columns

print(f"Audit size: {total_rows:,} rows, {len(cols)} columns")

# 1) Missing values per column (on full df_raw)
missing_counts_df = df_audit.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in cols
])

missing_profile_df = (
    missing_counts_df.toPandas()
    .T
    .reset_index()
    .rename(columns={"index": "Column Name", 0: "Missing Count"})
)

missing_profile_df["Missing Percentage"] = (
    missing_profile_df["Missing Count"] / total_rows * 100
).round(2)

# 2) dtypes snapshot (from Spark)
dtypes_profile_df = (
    spark.createDataFrame(
        [(c, str(t)) for c, t in df_audit.dtypes],
        schema=["Column Name", "Data Type"],
    )
    .toPandas()
)

# 3) Duplicate summary on the full df_raw (full-row equality)
dup_count = total_rows - df_audit.dropDuplicates().count()
dup_percentage = (
    round(dup_count / total_rows * 100, 4) if total_rows else 0.0
)

dup_summary_pdf = (
    spark.createDataFrame(
        [
            (
                total_rows,
                dup_count,
                dup_percentage,
                "Duplicates Found" if dup_count > 0 else "No Duplicates",
            )
        ],
        schema=[
            "Total Rows",
            "Duplicate Count",
            "Duplicate Percentage",
            "Status",
        ],
    )
    .toPandas()
)

print(
    "Baseline Audit BEFORE Cleaning — "
    f"Rows: {total_rows:,}, Columns: {len(cols):,} "
    "(full df_raw in Spark)."
)

display(missing_profile_df.sort_values("Missing Percentage", ascending=False))
display(dtypes_profile_df)
display(dup_summary_pdf)

# Optional: free cache if you don't need df_audit again
df_audit.unpersist()


With reference to the Code Cell above, we are able to determine the various actions that needs to be done within the Transform Process of the ETL Pipeline based of the quality of the provided Sensor Data.

---
# 3 Data Transform Layer

In this section, we will be conducting the Transform component of the ETL Process. The Transform process will include the various operations such as dropping duplicates and imputing missing data so as to clean and prepare the Data for the Data Warehouse. This layer will incur physical changes to the Sensor Data extracted from the Staging Database previously.

---
## 3.1 Duplicate Handling

In this sub-section, we will be addressing any duplicated values within the Sensor Data. Duplicated Data can cause serveral issues during Data Analysis and Machine Learning operations such as causing a change in numerical summaries despite the entry being illegitimate. Therefore, duplicated data must be handled correctly by dropping.

In [ ]:
"""
Cell S3 — Duplicate Handling (Spark, full-row duplicates ignoring csv_index)

- Operates directly on df_raw (Spark DataFrame).
- Treats rows as duplicates if ALL columns except `csv_index` are identical.
- Keeps the first occurrence of each unique combination of
  (booking_id, accuracy, bearing, ..., speed, cleaned_date).
- Preserves `csv_index` as a normal column (not part of the dedup key).
- Returns df_dedup / df_clean_base for downstream cleaning steps.
"""


# All columns in df_raw
all_cols = df_raw.columns

# Deduplication key = all columns EXCEPT csv_index
dedup_key_cols = [c for c in all_cols if c != "csv_index"]

print("Deduplication key columns (excluding csv_index):")
print(dedup_key_cols)

before = df_raw.count()

df_dedup = df_raw.dropDuplicates(subset=dedup_key_cols)
after = df_dedup.count()
n_dups = before - after

print(
    f"Dropped full-row duplicates ignoring `csv_index`: {n_dups:,} | "
    f"BEFORE — Rows: {before:,}, AFTER — Rows: {after:,}, Cols: {len(df_dedup.columns):,}"
)

# This becomes the base DF for subsequent cleaning steps
df_clean_base = df_dedup

With reference to the Code Cell above, we are able to determine that the Duplicated Data entry have been successfully dropped.

---
## 3.2 Data Types Casting

In this sub-section, we will be defining the various Data Types for each of the feature within the Sensor Data. Subsequently, we will force convert the entire feature to its respective Data Type and converting any non-compliant values to 'NaN' which will be addressed later on during the imputation of Missing Data.

In [ ]:
"""
Cell S4 — Data Types: Fixed-Schema Casting Plan (PySpark)

We enforce an explicit schema on df_dedup:

    • ID/Categorical:  booking_id  → string
    • Numeric:         accuracy, bearing,
                       acceleration_x, acceleration_y, acceleration_z,
                       gyro_x, gyro_y, gyro_z,
                       second, speed   → double

Casting uses Spark's .cast("double") semantics:
    - invalid values become NULL (no hard failures)
    - we also report how many new NULLs are introduced per column.

Inputs
------
df_dedup : Spark DataFrame
    Output of S3 (after duplicate handling).

Outputs
-------
df_cast : Spark DataFrame
    Typed version of df_dedup with enforced schema.
"""


# ------------------- Config: target schema -------------------

ID_COL = "booking_id"
NUM_COLS: List[str] = [
    "accuracy", "bearing",
    "acceleration_x", "acceleration_y", "acceleration_z",
    "gyro_x", "gyro_y", "gyro_z",
    "second", "speed",
]

# Only check columns that actually exist in df_dedup
cols_to_check = [c for c in [ID_COL] + NUM_COLS if c in df_dedup.columns]

# ------------------- 1) Null counts BEFORE casting -------------------

before_nulls_row = df_dedup.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in cols_to_check
]).collect()[0]

before_nulls: Dict[str, int] = {c: int(before_nulls_row[c]) for c in cols_to_check}

# Capture schema before casting
schema_before = {name: dtype for name, dtype in df_dedup.dtypes}

# ------------------- 2) Apply casts in Spark -------------------

df_cast = df_dedup

# ID as string (safe join key)
if ID_COL in df_cast.columns:
    df_cast = df_cast.withColumn(ID_COL, F.col(ID_COL).cast("string"))

# Numeric columns as double
for col in NUM_COLS:
    if col in df_cast.columns:
        df_cast = df_cast.withColumn(col, F.col(col).cast("double"))

# ------------------- 3) Null counts AFTER casting -------------------

after_nulls_row = df_cast.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in cols_to_check
]).collect()[0]

after_nulls: Dict[str, int] = {c: int(after_nulls_row[c]) for c in cols_to_check}

# Capture schema after casting
schema_after = {name: dtype for name, dtype in df_cast.dtypes}

# ------------------- 4) Build casting report (small, via pandas) -------------------

report_rows: List[Dict[str, object]] = []
for c in cols_to_check:
    report_rows.append({
        "Column": c,
        "From Type": schema_before.get(c, "N/A"),
        "To Type": schema_after.get(c, "N/A"),
        "New NULLs Added": after_nulls[c] - before_nulls[c],
        "NULLs Before": before_nulls[c],
        "NULLs After": after_nulls[c],
    })

dtype_cast_report = pd.DataFrame(report_rows)

print("Post-cast Spark dtypes:")
dtypes_df = pd.DataFrame(df_cast.dtypes, columns=["Column Name", "Data Type"])
display(dtypes_df)

print("\nCasting report:")
display(dtype_cast_report)

# Optional: make this the main working DF for next cells
df_typed = df_cast

With reference to the Code Cell above, we are able to determine that the various features within the Sensor Data have been casted with their respective appropriate Data Types.

---
## 3.3 Removing Invalid Numerical Values

In this section, we will be identifying and casting any invalid numerical values into NaN with reference to the business context and information derived from the Metadata. These missing values (if any) will subsequently be imputed in the next sub-section. By removing these values, it maintains the integrity and quality of the data provided.

In [ ]:
"""
Cell C6 — Rule-Based Validation (Accuracy, Bearing, Speed, Second) — Spark

Business rules:
    - Invalid Accuracy (< 0)             → set to NULL
    - Invalid Bearing (outside [0, 360]) → set to NULL
    - Invalid Speed (< 0)                → set to NULL
    - Invalid Second (< 0)               → set to NULL

Also:
    - Adds temporary flag columns:
        * accuracy_invalid (0/1)
        * bearing_invalid  (0/1)
        * speed_invalid    (0/1)
        * second_invalid   (0/1)
        * row_any_invalid  (0/1 if any of the above is 1)
    - Prints a summary of how many values/rows were flagged.
    - Drops the helper flag columns at the end.

Inputs
------
df_typed : Spark DataFrame
    Output of S4 (typed schema: numeric cols as double).

Outputs
-------
df_validated : Spark DataFrame
    Same columns as df_typed, but with invalid values set to NULL.
"""

EPS = 1e-12  # small tolerance for floating-point comparisons

# Start from the typed DF (from S4)
df_validated = df_typed

# Ensure numeric types (defensive; S4 already cast them)
for c in ["accuracy", "bearing", "speed", "second"]:
    if c in df_validated.columns:
        df_validated = df_validated.withColumn(c, F.col(c).cast("double"))

# ---- Accuracy invalid (< 0) -----------------------------------------------
if "accuracy" in df_validated.columns:
    df_validated = (
        df_validated
        .withColumn(
            "accuracy_invalid",
            F.when(
                (F.col("accuracy").isNotNull()) & (F.col("accuracy") < -EPS),
                F.lit(1),
            ).otherwise(F.lit(0))
        )
        .withColumn(
            "accuracy",
            F.when(F.col("accuracy_invalid") == 1, F.lit(None)).otherwise(F.col("accuracy"))
        )
    )
else:
    df_validated = df_validated.withColumn("accuracy_invalid", F.lit(0))

# ---- Bearing invalid (not in [0, 360]) ------------------------------------
if "bearing" in df_validated.columns:
    df_validated = (
        df_validated
        .withColumn(
            "bearing_invalid",
            F.when(
                (F.col("bearing").isNotNull()) &
                ((F.col("bearing") < 0 - EPS) | (F.col("bearing") > 360 + EPS)),
                F.lit(1),
            ).otherwise(F.lit(0))
        )
        .withColumn(
            "bearing",
            F.when(F.col("bearing_invalid") == 1, F.lit(None)).otherwise(F.col("bearing"))
        )
    )
else:
    df_validated = df_validated.withColumn("bearing_invalid", F.lit(0))

# ---- Speed invalid (< 0) --------------------------------------------------
if "speed" in df_validated.columns:
    df_validated = (
        df_validated
        .withColumn(
            "speed_invalid",
            F.when(
                (F.col("speed").isNotNull()) & (F.col("speed") < -EPS),
                F.lit(1),
            ).otherwise(F.lit(0))
        )
        .withColumn(
            "speed",
            F.when(F.col("speed_invalid") == 1, F.lit(None)).otherwise(F.col("speed"))
        )
    )
else:
    df_validated = df_validated.withColumn("speed_invalid", F.lit(0))

# ---- Second invalid (< 0) -------------------------------------------------
if "second" in df_validated.columns:
    df_validated = (
        df_validated
        .withColumn(
            "second_invalid",
            F.when(
                (F.col("second").isNotNull()) & (F.col("second") < -EPS),
                F.lit(1),
            ).otherwise(F.lit(0))
        )
        .withColumn(
            "second",
            F.when(F.col("second_invalid") == 1, F.lit(None)).otherwise(F.col("second"))
        )
    )
else:
    df_validated = df_validated.withColumn("second_invalid", F.lit(0))

# ---- Row-level invalid helper ---------------------------------------------
df_validated = df_validated.withColumn(
    "row_any_invalid",
    (
        F.col("accuracy_invalid")
        + F.col("bearing_invalid")
        + F.col("speed_invalid")
        + F.col("second_invalid")
        > 0
    ).cast("int")
)

# ---- Summaries ------------------------------------------------------------
summary_pdf = df_validated.agg(
    F.sum("accuracy_invalid").alias("invalid_accuracy_count"),
    F.sum("bearing_invalid").alias("invalid_bearing_count"),
    F.sum("speed_invalid").alias("invalid_speed_count"),
    F.sum("second_invalid").alias("invalid_second_count"),
    F.sum("row_any_invalid").alias("invalid_rows_any"),
    F.count(F.lit(1)).alias("total_rows"),
).toPandas()

print("Validation summary (accuracy / bearing / speed / second rules):")
display(summary_pdf)

# ---- Optional cleanup: drop helper columns --------------------------------
df_validated = df_validated.drop(
    "accuracy_invalid",
    "bearing_invalid",
    "speed_invalid",
    "second_invalid",
    "row_any_invalid",
)

With reference to the Code Cell above, we are able to confirm that there are no abnormalies within the Sensor Dataset provided.

---
## 3.4 Missing Data Imputation

In this sub-section, we will be imputing the Missing Data using a Machine Learning algorithm. The Machine Learning algorithm in use is and IterativeImputer with ExtraTreesRegressor. The ExtraTreesRegressor is an ensemble of randomised trees which is capable of capturing non-linear relationships and various Feature Interactions without heavy reliance on parameter tuning. This makes the algorithm very suitable for Sensor Data in our case. Through using a Machine Learning algorithm to impute a data, we are not replacing the value with a 'stable value' such as mean or median, rather we hope to achieve the actual desired value that was not shown in the dataset due to multiple possible reasons.

---
### 3.4.1 Data Preparation for Imputation

In this sub-sub-section, we will be preparing our data for the imputation using the Machine Learning alogrithm. We will be mainly handling the data rows with missing data so as to allow for a clean and substential data for the ExtraTreesRegressor to train and impute the missing data.

In [ ]:
"""
Cell C5 — NA Pruning Before Imputation (Spark)

What this does (on df_validated):
  1) Drop 'cleaned_date' (we don't impute it).
  2) Drop rows where ALL numeric cols are NULL.
  3) Drop numeric columns that are entirely NULL (no observed non-null values).

Inputs
------
df_validated : Spark DataFrame
    Output of C6 (after rule-based validation).

Outputs
-------
df_prep : Spark DataFrame
    Pruned DataFrame ready for imputation.
"""

from pyspark.sql import functions as F

numeric_cols = [
    "accuracy", "bearing",
    "acceleration_x", "acceleration_y", "acceleration_z",
    "gyro_x", "gyro_y", "gyro_z",
    "second", "speed",
]

df_prep = df_validated

# 0) Drop cleaned_date (we don't impute it)
if "cleaned_date" in df_prep.columns:
    df_prep = df_prep.drop("cleaned_date")
    print("Dropped 'cleaned_date' before imputation.")

# 1) Drop all-NULL rows across numeric columns
null_exprs = [F.col(c).isNull() for c in numeric_cols if c in df_prep.columns]

if null_exprs:
    all_null_cond = null_exprs[0]
    for expr in null_exprs[1:]:
        all_null_cond = all_null_cond & expr

    before_rows = df_prep.count()
    df_prep = df_prep.filter(~all_null_cond)
    after_rows = df_prep.count()
    print("Dropped all-NULL numeric rows:", before_rows - after_rows)
else:
    print("No numeric columns found for all-NULL row pruning.")

# 2) Drop numeric columns that are entirely NULL
cols_present = df_prep.columns
numeric_present = [c for c in numeric_cols if c in cols_present]

to_drop = []
for c in numeric_present:
    non_null = df_prep.filter(F.col(c).isNotNull()).limit(1).count()
    if non_null == 0:
        to_drop.append(c)

if to_drop:
    df_prep = df_prep.drop(*to_drop)
    print("Dropped numeric cols with zero observed values:", to_drop)
else:
    print("No numeric columns were entirely NULL.")

print("After pruning:", df_prep.count(), "rows,", len(df_prep.columns), "cols")

With reference to the Code Cell above, we are able to verify that the missing data entry within the Sensor Data have been dropped successfully and we have a set of data to train the ExtraTreeRegressor with the IterativeImputer in this next Code Cell.

---
### 3.4.2 Machine Learning Imputation of Missing Data

In this sub-sub-section, we will be utilising the previously parsed Data to conduct the Imputation using the ExtraTreesRegressor with the IterativeImputer. This will allow us to attempt to impute data to what it is originally intended to be despite it being missing. Should there be a lack of data due to most if not all data being dropped, we will be utilising the k-Nearest Neighbour to impute the data instead. This operation will be done in the Code Cell below.

In [ ]:
"""
Cell C6 — Missing Data Imputation via Spark ML Imputer (Median)

- Operates on df_prep (Spark DataFrame from C5).
- Uses pyspark.ml.feature.Imputer to fill numeric columns with the median.
- booking_id is left untouched, even if it has a numeric dtype.
- Output: df_imputed (Spark DataFrame) with the same column names as df_prep.
"""

# 1) Detect numeric columns eligible for imputation
numeric_present = [
    c for c, t in df_prep.dtypes
    if t in ("double", "float", "int", "bigint") and c != "booking_id"
]

print("Numeric columns to impute:", numeric_present)

if numeric_present:
    # 2) Configure Spark ML Imputer (median strategy)
    imputer = Imputer(
        inputCols=numeric_present,
        outputCols=[f"{c}__imputed" for c in numeric_present],
        strategy="median",
    )

    # 3) Fit on df_prep and transform
    model = imputer.fit(df_prep)
    df_imputed_tmp = model.transform(df_prep)

    # 4) Replace original numeric columns with their imputed versions
    for c in numeric_present:
        df_imputed_tmp = (
            df_imputed_tmp
            .drop(c)
            .withColumnRenamed(f"{c}__imputed", c)
        )
else:
    print("No numeric columns detected for imputation. Skipping Imputer step.")
    df_imputed_tmp = df_prep

# 5) Final imputed DF
df_imputed = df_imputed_tmp

# 6) Quick sanity check: remaining NULLs per column
print("Imputation complete. Remaining NULLs per column:")
df_imputed.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in df_imputed.columns
]).show(truncate=False)

With reference to the Code Cell above, we are able to verify that the Missing Data have been successfully imputed and we are able to proceed with identifying the various outliers and abnormalies within our Sensor Data.

---
## 3.5 Outliers Identification and Addressment

In this sub-section, we will be conducting the Outliers Identification and Addressment. Outliers can often skew Machine Learning and Data Visualisation due to their extreme value (either extremely high or extremely low). This can cause numerical summaries and the mathematical and design principal behind various Machine Learning algorithm to be obsolete. Therefore, identifying and addressing outliers are extremely important but we should not ignore the fact that there is legitimate values in the Outliers.

---
### 3.5.1 Univariate Outliers Detection

In this sub-section, we will be executing the Univariate Outlier Detection Function. The Univariate Outlier Detection will be used for each feature and each features are independent of each other. The methodology utilised for the Outlier Dectection are Inter-Quartile Range, Median Absolute Deviation and Z-Score. The relevant mathematical formulas are indicated below:

---
**Interquartile Range (IQR) Rule**

Purpose: Flags Points far from the Central 50% of the data

$$\text{IQR} = Q_3 - Q_1$$

$$\text{Lower Bound} = Q_1 - k \times \text{IQR}$$

$$\text{Upper Bound} = Q_3 + k \times \text{IQR}$$

**Legend:**
- $Q_1$ = 25th Percentile
- $Q_3$ = 75th Percentile
- $\text{IQR}$ = Interquartile range
- $k$ = IQR Multiplier (commonly 1.5 or 3.0)

---
**Median Absolute Deviation (MAD)**

Purpose: Flags Points far from the Central 50% of the data

$$\text{MAD} = \text{median}\left(|x_i - \tilde{x}|\right)$$

$$z_i = \frac{0.6745 \times (x_i - \tilde{x})}{\text{MAD}}$$

$$\text{Outlier if } |z_i| > T_{\text{MAD}}$$

**Legend:**
- $Q_1$ = 25th Percentile
- $Q_3$ = 75th Percentile
- $\text{IQR}$ = Interquartile range
- $k$ = IQR Multiplier (commonly 1.5 or 3.0)

In [ ]:
"""
Cell C7 — Univariate Outlier Flagging (Spark, IQR)

- For each numeric column, compute IQR and flag points outside:
      [Q1 - 1.5 * IQR, Q3 + 1.5 * IQR]
- Adds boolean columns: "<col>__is_uni_outlier"

Inputs
------
df_imputed : Spark DataFrame
    Output of C6 (after median imputation).

Outputs
-------
df_uni_flagged : Spark DataFrame
    df_imputed + one boolean flag column per numeric feature.
"""

# 1) Identify numeric columns (exclude booking_id)
numeric_for_uni = [
    c for c, t in df_imputed.dtypes
    if t in ("double", "float", "int", "bigint") and c != "booking_id"
]

print("Numeric columns for univariate IQR outlier check:", numeric_for_uni)

df_uni = df_imputed
total_rows = df_uni.count()

uni_summary_rows = []

# 2) Loop through each numeric column and apply IQR rule
for c in numeric_for_uni:
    # approxQuantile returns [q1, q3]
    q1, q3 = df_uni.approxQuantile(c, [0.25, 0.75], 0.001)
    iqr = q3 - q1

    if iqr == 0:
        # No variability => no outliers
        df_uni = df_uni.withColumn(f"{c}__is_uni_outlier", F.lit(False))
        uni_summary_rows.append((c, "iqr", 0, 0.0))
        continue

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    flag_col_name = f"{c}__is_uni_outlier"
    flag_expr = F.when(
        (F.col(c) < lower) | (F.col(c) > upper),
        F.lit(True)
    ).otherwise(F.lit(False))

    df_uni = df_uni.withColumn(flag_col_name, flag_expr)

    n_out = df_uni.filter(F.col(flag_col_name) == True).count()
    pct_out = (n_out / total_rows * 100.0) if total_rows else 0.0

    uni_summary_rows.append((c, "iqr", n_out, pct_out))

# 3) Build summary table (Spark → small pandas for display)
uni_summary_spark = spark.createDataFrame(
    uni_summary_rows,
    schema=["column", "method", "n_outliers", "pct_outliers"],
)

uni_summary = uni_summary_spark.orderBy(F.desc("pct_outliers")).toPandas()

print("Univariate outliers flagged (IQR).")
display(uni_summary.head(12))

# 4) Hand-off
df_uni_flagged = df_uni

With reference to the Code Cell above, we are able to verify the various methods undertaken to address the Outliers in the Sensor Data at the Univariate level.

---
### 3.5.3 Multivariate Outlier Detection

In this sub-section, we will be executing the Multivariate Outlier Detection Function. The Multivariate Outlier Detection will be used for the entire Sensor Data acorss all the various features. The methodology utilised for the Outlier Dectection are Isolation Forest, Elliptic Envelope and Local Outlier Factor. The relevant mathematical formulas are indicated below:

In [ ]:
"""
Cell C8 — Multivariate Outlier Flagging (Spark, PCA Reconstruction)

Approach
--------
1) Use numeric features (excluding booking_id) to build a feature vector.
2) Standardize features (zero mean, unit variance) using StandardScaler.
3) Fit PCA with k components (k <= 3, capped by number of features).
4) Compute reconstruction error in scaled space:
       error = || x_scaled - x_hat ||^2
       where x_hat = PC * z,  z = PCA projection.
5) Normalise error to [0, 1] as `_mv_score`.
6) Flag top X% (contamination) as multivariate outliers:
       `_mv_is_outlier` = True/False.

Inputs
------
df_uni_flagged : Spark DataFrame
    Output from C7 with univariate outlier flags.

Outputs
-------
df_clean : Spark DataFrame
    df_uni_flagged + `_mv_score` + `_mv_is_outlier`
"""

# 0) Select numeric columns for multivariate detection
numeric_mv = [
    c for c, t in df_uni_flagged.dtypes
    if t in ("double", "float", "int", "bigint") and c != "booking_id"
]

print("Numeric columns for MV PCA detection:", numeric_mv)

if not numeric_mv:
    # No numeric features → nothing to do
    df_clean = (
        df_uni_flagged
        .withColumn("_mv_score", F.lit(0.0))
        .withColumn("_mv_is_outlier", F.lit(False))
    )
    print("No numeric columns for MV detection; `_mv_score` set to 0, no MV outliers flagged.")
else:
    # 1) Build feature vector
    assembler = VectorAssembler(
        inputCols=numeric_mv,
        outputCol="features",
        handleInvalid="skip",  # rows with invalids get skipped from vectorisation
    )
    df_vec = assembler.transform(df_uni_flagged)

    # 2) Standardise features
    scaler = StandardScaler(
        inputCol="features",
        outputCol="features_scaled",
        withMean=True,
        withStd=True,
    )
    scaler_model = scaler.fit(df_vec)
    df_scaled = scaler_model.transform(df_vec)

    # 3) PCA in scaled feature space
    d = len(numeric_mv)
    k = min(d, 3) if d > 0 else 0

    if k == 0:
        # Degenerate case: only 1 numeric col or similar
        df_tmp = (
            df_scaled
            .withColumn("_mv_score", F.lit(0.0))
            .withColumn("_mv_is_outlier", F.lit(False))
        )
        # Clean temp vector columns if they exist
        for tmp_col in ["features", "features_scaled"]:
            if tmp_col in df_tmp.columns:
                df_tmp = df_tmp.drop(tmp_col)
        df_clean = df_tmp
        print("k=0 for PCA; `_mv_score` set to 0, no MV outliers flagged.")
    else:
        pca = PCA(
            k=k,
            inputCol="features_scaled",
            outputCol="pca_features",
        )
        pca_model = pca.fit(df_scaled)

        # This was missing: actually create `pca_features`
        df_pca = pca_model.transform(df_scaled)

        pc = pca_model.pc  # DenseMatrix of shape (d, k)

        # Broadcast PC to executors
        pc_bc = spark.sparkContext.broadcast(pc)

        @F.udf(T.DoubleType())
        def recon_error(x_scaled, z):
            """
            Compute squared reconstruction error ||x_scaled - x_hat||^2
            in scaled feature space using PCA components.
            """
            if x_scaled is None or z is None:
                return None

            pc_local = pc_bc.value  # DenseMatrix
            d_local, k_local = pc_local.numRows, pc_local.numCols

            # Convert Spark vectors to numpy arrays
            try:
                x_arr = np.array(x_scaled.toArray(), dtype=float)
                z_arr = np.array(z.toArray(), dtype=float)
            except AttributeError:
                # Fallback if already arrays/list-like
                x_arr = np.array(x_scaled, dtype=float)
                z_arr = np.array(z, dtype=float)

            # Guard shapes
            if x_arr.shape[0] != d_local or z_arr.shape[0] != k_local:
                return None

            # x_hat = PC * z
            x_hat = np.zeros(d_local, dtype=float)
            for col_idx in range(k_local):
                zval = float(z_arr[col_idx])
                if zval == 0.0:
                    continue
                for row_idx in range(d_local):
                    x_hat[row_idx] += pc_local[row_idx, col_idx] * zval

            diff = x_arr - x_hat
            return float(np.dot(diff, diff))

        # 4) Score each row with reconstruction error
        df_scored = df_pca.withColumn(
            "_mv_score_raw",
            recon_error(F.col("features_scaled"), F.col("pca_features")),
        )

        # 5) Normalise scores to [0, 1]
        stats_row = df_scored.agg(
            F.min("_mv_score_raw").alias("min_s"),
            F.max("_mv_score_raw").alias("max_s"),
        ).collect()[0]

        min_s = stats_row["min_s"]
        max_s = stats_row["max_s"]

        if min_s is None or max_s is None:
            # All scores NULL? Fallback: no MV signal
            df_tmp = (
                df_scored
                .withColumn("_mv_score", F.lit(0.0))
                .withColumn("_mv_is_outlier", F.lit(False))
            )
            for tmp_col in ["features", "features_scaled", "pca_features", "_mv_score_raw"]:
                if tmp_col in df_tmp.columns:
                    df_tmp = df_tmp.drop(tmp_col)
            df_clean = df_tmp
            print("All MV scores are NULL; treating as no MV outliers.")
        else:
            denom = (max_s - min_s) if max_s != min_s else 1e-12
            df_scored = df_scored.withColumn(
                "_mv_score",
                (F.col("_mv_score_raw") - F.lit(min_s)) / F.lit(denom),
            )

            # 6) Determine contamination rate and threshold
            total_mv = df_scored.count()
            if total_mv < 500:
                contamination = 0.05
            elif total_mv < 5000:
                contamination = 0.02
            else:
                contamination = 0.01

            thr = df_scored.approxQuantile(
                "_mv_score_raw",
                [1.0 - contamination],
                0.001,
            )[0]

            # 7) Flag MV outliers and clean temp columns
            df_tmp = (
                df_scored
                .withColumn(
                    "_mv_is_outlier",
                    (F.col("_mv_score_raw") >= F.lit(thr)).cast("boolean"),
                )
            )

            for tmp_col in ["features", "features_scaled", "pca_features", "_mv_score_raw"]:
                if tmp_col in df_tmp.columns:
                    df_tmp = df_tmp.drop(tmp_col)

            df_clean = df_tmp

            print("MV detection via PCA reconstruction")
            print("Contamination:", contamination, "| threshold_raw:", thr)
            print(
                "MV outliers flagged:",
                df_clean.filter(F.col("_mv_is_outlier") == True).count(),
            )


With reference to the Code Cell above, we are able to verify the various methods undertaken to address the Outliers in the Sensor Data at the Multivariate level.

## 3.6 Feature Engineering

We convert high-frequency sensor readings into per-trip (booking_id) features that summarize driving style, stability, and potential risk signals. These features support EDA, dashboard KPIs, and modeling (in CA2).

All computations in this section are implemented using **PySpark** DataFrame operations and window functions (instead of pandas), but the mathematical definitions remain identical to our original design.

### Row-level Signals

- `acc_magnitude = sqrt(ax² + ay² + az²)`: collapses 3-axis acceleration into a single “force” measure. Useful to spot harsh events even when sign/orientation differs across devices.

- `gyro_magnitude = sqrt(gx² + gy² + gz²)`: total angular velocity; spikes indicate fast rotations (e.g., swerves).

### Trip-level Aggregations

- **Speed profile**: `speed_mean`, `speed_std`, `speed_max`, `speed_min`, `speed_p90`.

  - Why: Average and spread quantify typical pace vs. variability; high p90 or max may indicate risky speeding bursts.

- **Acceleration profile**: `acc_mag_mean`, `acc_mag_std`, `acc_mag_p95`, `acc_mag_max`.

  - Why: Captures overall driving smoothness and rare-but-strong events (95th percentile and max).

- **Axis-level accelerations**: `acc_x_mean/std`, `acc_y_mean/std`, `acc_z_mean/std`.

  - Why: Aids interpretation: longitudinal (x) relates to braking/accel; lateral (y) to cornering; vertical (z) to bumps/road quality.

- **Jerk (abruptness)**: `jerk_med`, `jerk_p95` from |Δacc_mag|/Δt.

  - Why: Sudden changes correlate with harsh braking/acceleration maneuvers often reported in safe-driving literature.

- **Turning behavior**: `turn_delta_std_deg` and `turn_rate_med_deg_s` from bearing changes.

  - Why: High heading volatility or high median turn rate flags sharp turns, lane weaving, or roundabout entry/exit aggressiveness.

- **Gyro profile**: `gyro_mag_mean`, `gyro_mag_std`, `gyro_mag_p95`.

  - Why: Rotation stability complements bearing; gyro can be more robust when GPS bearing is noisy at low speeds.

- **Trip meta**: `trip_duration_s`, `n_samples`, `sampling_rate_hz`.

  - Why: Contextualizes feature magnitudes (short vs. long trips) and helps detect sensor sampling anomalies.

- **Start/End speed**: `start_speed`, `end_speed`.

  - Why: Quick sanity checks (e.g., trips starting/ending at rest), and can hint at aggressive starts/stops.

- **GPS accuracy**: `gps_acc_mean`, `gps_acc_std`.

  - Why: Downstream modeling can down-weight noisy trips or diagnose patterns driven by poor GPS quality.


### 3.6.1 Setup, hygiene, and sorting

**Purpose.** Prepare a clean, analysis-ready copy of `df_cleaned` (now a Spark DataFrame), coerce numeric fields, and ensure chronological ordering within each trip for time-based calculations.

**What this cell does**
- **Work on a derived DataFrame**: Uses `dfc = df_cleaned` as the working frame so that upstream cells can still refer to the original `df_cleaned` if needed.
- **Type safety**: Casts sensor columns (e.g., `speed`, `acceleration_x`, `gyro_z`) and supporting fields (`accuracy`, `bearing`, `second`) to numeric Spark types. Invalid strings become `NULL`, which downstream aggregations can safely ignore.
- **Temporal ordering**: Uses `booking_id` + `second` ordering in window functions so that per-trip differences (e.g., |Δspeed|/Δt, jerk) are computed along the **true time sequence**.

**Why this matters for detecting dangerous driving**
- Time derivatives (e.g., speed change rate, jerk) **require correct order**; misordered rows will produce false spikes.
- Consistent numeric types prevent subtle string/float mixups that break aggregations or generate misleading outliers.

**Notes**
- We intentionally **do not touch** `cleaned_date` here; it is handled separately in the staging update step.
- We rely on PySpark’s `cast` semantics (`invalid -> NULL`) instead of `pd.to_numeric(errors="coerce")`, but the effect is equivalent.


In [ ]:
"""
Cell 1 — Setup & Hygiene (PySpark)

Purpose
-------
Create a clean working copy `dfc` from df_clean, with:
    - Numeric sensor columns cast to double.
    - booking_id cast to string.
    - Optional ordering by booking_id, second for inspection.

Inputs
------
df_clean : Spark DataFrame
    Output of C8 (after all cleaning & outlier flagging).

Outputs
-------
dfc : Spark DataFrame
    Sanitised working DataFrame for downstream feature engineering.
"""


# Start from the cleaned DF from previous step
dfc = df_clean

# Numeric columns we expect (cast defensively)
_NUM_COLS = [
    "accuracy",
    "bearing",
    "acceleration_x",
    "acceleration_y",
    "acceleration_z",
    "gyro_x",
    "gyro_y",
    "gyro_z",
    "second",
    "speed",
]

# Ensure numeric columns are double where present
for col in _NUM_COLS:
    if col in dfc.columns:
        dfc = dfc.withColumn(col, F.col(col).cast("double"))

# Ensure booking_id exists and is consistently string
if "booking_id" in dfc.columns:
    dfc = dfc.withColumn("booking_id", F.col("booking_id").cast("string"))

# Optional: create a globally sorted view for inspection / later window ops
_sort_cols = [c for c in ("booking_id", "second") if c in dfc.columns]
if _sort_cols:
    dfc = dfc.orderBy(*_sort_cols)

print("Cell 1 complete — dfc schema:")
dfc.printSchema()

### 3.6.2 Row-level derived signals

**Purpose.** Create **instantaneous magnitudes** that summarize 3-axis motion at each timestamp:
- `acc_magnitude = sqrt(ax² + ay² + az²)`
- `gyro_magnitude = sqrt(gx² + gy² + gz²)`

**Why we compute these**
- A single magnitude per row captures **overall linear force** (acceleration) and **rotational intensity** (gyro), regardless of device orientation.
- These row-level signals are later summarized into per-trip statistics (mean, std, max, p95), which align with **harsh braking**, **sudden acceleration**, **swerves**, and **sharp cornering**.

**Implementation details**
- Columns created here are **temporary** (no `FE_` prefix) because they are intermediate inputs to the per-trip aggregator.
- We use PySpark column expressions (`F.sqrt`, `F.col`²) instead of NumPy, so the code scales to large sensor logs.


In [ ]:
# ---------------- Magnitudes ----------------

dfc = dfc.withColumn(
    "acc_magnitude",
    F.sqrt(
        F.col("acceleration_x") ** 2
        + F.col("acceleration_y") ** 2
        + F.col("acceleration_z") ** 2
    )
)

dfc = dfc.withColumn(
    "gyro_magnitude",
    F.sqrt(
        F.col("gyro_x") ** 2
        + F.col("gyro_y") ** 2
        + F.col("gyro_z") ** 2
    )
)

print("Cell F1 complete — added acc_magnitude and gyro_magnitude.")
dfc.select(
    "booking_id", "second", "acc_magnitude", "gyro_magnitude"
).show(10, truncate=False)

### 3.6.3 Time-derivative and turning signals (Spark window-based)

**Purpose.** Derive robust, per-row time-series signals needed for our trip-level features:

- `delta_speed`, `delta_t` and `speed_change_rate = |Δspeed| / Δt`
- `delta_accm` and `jerk = |Δacc_magnitude| / Δt`
- `bearing_diff_deg` and `turn_rate_deg_s` as proxies for turning intensity

**Why they matter**
- **Reliable rates**: Derivatives like speed change rate and jerk are only meaningful if we divide by a non-zero Δt and ignore invalid values.
- **Turning behaviour**: Bearing wraps at 360°, so we use the minimal angle difference (shortest arc) as an efficient approximation of true heading change.

**Implementation details**
- We use a `Window.partitionBy("booking_id").orderBy("second")` to ensure all differences are computed along the correct time order within each trip.
- Zero or negative Δt values are treated as invalid and produce `NULL` rates, mirroring the protections in our original NumPy helpers.


In [ ]:
"""
Cell F2 — Time-Derivative & Turning Signals (PySpark)

Purpose
-------
Per booking_id (trip), compute:
    - delta_speed       = v_t - v_{t-1}
    - delta_t           = t - t_{t-1}
    - speed_change_rate = |delta_speed| / delta_t
    - delta_accm        = acc_magnitude_t - acc_magnitude_{t-1}
    - jerk              = |delta_accm| / delta_t
    - bearing_diff_deg  = minimal heading change in degrees (0–180)
    - turn_rate_deg_s   = bearing_diff_deg / delta_t
"""


w_trip_time = Window.partitionBy("booking_id").orderBy("second")

# Δspeed and Δt
dfc = dfc.withColumn(
    "delta_speed",
    F.col("speed") - F.lag("speed").over(w_trip_time)
).withColumn(
    "delta_t",
    F.col("second") - F.lag("second").over(w_trip_time)
)

# Speed change rate |Δv| / Δt
dfc = dfc.withColumn(
    "speed_change_rate",
    F.when(F.col("delta_t") > 0, F.abs(F.col("delta_speed") / F.col("delta_t")))
)

# Jerk from acceleration magnitude: |Δacc_magnitude| / Δt
dfc = dfc.withColumn(
    "delta_accm",
    F.col("acc_magnitude") - F.lag("acc_magnitude").over(w_trip_time)
).withColumn(
    "jerk",
    F.when(F.col("delta_t") > 0, F.abs(F.col("delta_accm") / F.col("delta_t")))
)

# Bearing-based turning:
# Use minimal angle difference to approximate unwrapped heading changes.
dfc = dfc.withColumn(
    "bearing_diff_raw",
    F.abs(F.col("bearing") - F.lag("bearing").over(w_trip_time))
).withColumn(
    "bearing_diff_deg",
    F.when(F.col("bearing_diff_raw") > 180, 360 - F.col("bearing_diff_raw"))
     .otherwise(F.col("bearing_diff_raw"))
)

dfc = dfc.withColumn(
    "turn_rate_deg_s",
    F.when(F.col("delta_t") > 0, F.col("bearing_diff_deg") / F.col("delta_t"))
)

print("Cell F2 complete — added time-derivative and turning signals.")
dfc.select(
    "booking_id", "second", "speed", "delta_speed", "delta_t",
    "speed_change_rate", "jerk", "bearing_diff_deg", "turn_rate_deg_s"
).show(10, truncate=False)

### 3.6.4 Per-Trip Feature Builder

Below are the formulas for the engineered features returned by `build_trip_features`.  
Feature names are shown as code (e.g., `FE_speed_mean`), while the math uses symbols for clarity.

---

#### Symbols (per trip)
- $i = 1,\dots,N$: index of ordered samples within a trip  
- $t_i$: timestamp (seconds)  
- $v_i$: speed  
- $a_{x,i}, a_{y,i}, a_{z,i}$: acceleration components  
- $g_{x,i}, g_{y,i}, g_{z,i}$: gyroscope components  
- $\theta_i$: **unwrapped** bearing angle (degrees)  
- $q_i$: GPS accuracy  
- $\bar{x}$: mean of $x$ over the trip  
- $P_{p}(x)$: $p$-th percentile of $x$ over the trip  
- $\Delta x_i = x_{i+1} - x_i$

---

## 1) Meta

- `FE_n_samples`:  
$$
N
$$

- `FE_trip_duration_s`:  
$$
t_N - t_1
$$

- `FE_sampling_rate_hz`:  
$$
\frac{N}{\,t_N - t_1\,}
\quad \text{(defined only if } t_N > t_1 \text{)}
$$

---

## 2) Speed Behavior

- `FE_speed_mean`, `FE_speed_std`:
$$
\overline{v}=\frac{1}{N}\sum_{i=1}^{N} v_i,
\qquad
\mathrm{std}(v)=\sqrt{\frac{1}{N}\sum_{i=1}^{N}(v_i-\overline{v})^2}
$$

- `FE_speed_max`, `FE_speed_min`, `FE_speed_p90`:
$$
\max(v_i), \quad \min(v_i), \quad P_{90}(v_i)
$$

- `FE_speed_change_rate_med` (proxy for harsh accel/brake):
$$
r_i=\frac{\,|\Delta v_i|\,}{\,\Delta t_i\,}
\quad\text{with}\quad \Delta v_i=v_{i+1}-v_i,\; \Delta t_i=t_{i+1}-t_i,
\qquad
\mathrm{median}(r_i)
$$

---

## 3) Acceleration Magnitude & Axis Context

Define acceleration magnitude per sample:
$$
a_i=\sqrt{a_{x,i}^2+a_{y,i}^2+a_{z,i}^2}
$$

- `FE_acc_mag_mean`, `FE_acc_mag_std`, `FE_acc_mag_max`, `FE_acc_mag_p95`:
$$
\overline{a}=\frac{1}{N}\sum_{i=1}^{N}a_i,\quad
\mathrm{std}(a)=\sqrt{\frac{1}{N}\sum_{i=1}^{N}(a_i-\overline{a})^2},\quad
\max(a_i),\quad P_{95}(a_i)
$$

Axis-level means and stds:

- `FE_acc_x_mean`, `FE_acc_y_mean`, `FE_acc_z_mean`:
$$
\overline{a_x}=\frac{1}{N}\sum a_{x,i},\quad
\overline{a_y}=\frac{1}{N}\sum a_{y,i},\quad
\overline{a_z}=\frac{1}{N}\sum a_{z,i}
$$

- `FE_acc_x_std`, `FE_acc_y_std`, `FE_acc_z_std`:
$$
\mathrm{std}(a_x),\quad \mathrm{std}(a_y),\quad \mathrm{std}(a_z)
= \sqrt{\frac{1}{N}\sum (a_{\cdot,i}-\overline{a_{\cdot}})^2}
$$

---

## 4) Jerk (abruptness from acceleration magnitude)

Per-step jerk:
$$
j_i=\frac{\,|a_{i+1}-a_i|\,}{\,t_{i+1}-t_i\,}
$$

- `FE_jerk_med`, `FE_jerk_p95`:
$$
\mathrm{median}(j_i),\qquad P_{95}(j_i)
$$

---

## 5) Gyroscope (rotational intensity)

Define gyro magnitude:
$$
g_i=\sqrt{g_{x,i}^2+g_{y,i}^2+g_{z,i}^2}
$$

- `FE_gyro_mag_mean`, `FE_gyro_mag_std`, `FE_gyro_mag_p95`:
$$
\overline{g}=\frac{1}{N}\sum g_i,\quad
\mathrm{std}(g)=\sqrt{\frac{1}{N}\sum(g_i-\overline{g})^2},\quad
P_{95}(g_i)
$$

---

## 6) Turning & Stability (bearing-based)

After **unwrapping** bearing to remove 360° wrap:

Per-step heading change and turn rate:
$$
\Delta\theta_i=|\theta_{i+1}-\theta_i|,\qquad
\omega_i=\frac{\Delta\theta_i}{\,t_{i+1}-t_i\,}
$$

- `FE_turn_delta_std_deg` (turning volatility):
$$
\mathrm{std}(\Delta\theta)=\sqrt{\frac{1}{N-1}\sum_{i=1}^{N-1}\left(\Delta\theta_i-\overline{\Delta\theta}\right)^2}
$$

- `FE_turn_rate_med_deg_s`:
$$
\mathrm{median}(\omega_i)
$$

---

## 7) GPS Accuracy & Trip Edges

- `FE_gps_acc_mean`, `FE_gps_acc_std`:
$$
\overline{q}=\frac{1}{N}\sum q_i,\qquad
\mathrm{std}(q)=\sqrt{\frac{1}{N}\sum(q_i-\overline{q})^2}
$$

- `FE_start_speed`, `FE_end_speed`:
$$
v_1,\qquad v_N
$$

---

**Notes**
- All aggregations are computed over **finite** values only (NaN-robust).  
- Per-step rates exclude samples where $\Delta t_i = 0$.  
- Bearing is **unwrapped** before computing $\Delta\theta_i$ to avoid false 360° jumps.

### Cell 4 — Per-trip aggregations in PySpark

**Purpose.** Implement the `build_trip_features` logic using Spark aggregations instead of a pandas `groupby().apply`.

**What this cell does**
- Groups the row-level `dfc` by `booking_id`.
- Aggregates all row-level signals into trip-level `FE_` features:
  - Speed profile (`FE_speed_*`)
  - Acceleration magnitude and axis-wise stats (`FE_acc_*`)
  - Jerk quantiles (`FE_jerk_med`, `FE_jerk_p95`)
  - Gyroscope profile (`FE_gyro_mag_*`)
  - Turning behaviour (`FE_turn_delta_std_deg`, `FE_turn_rate_med_deg_s`)
  - GPS accuracy (`FE_gps_acc_*`)
  - Trip meta (`FE_n_samples`, `FE_trip_duration_s`, `FE_sampling_rate_hz`)

The formulas match the 3.6.4 definitions, but are executed fully inside Spark for scalability.


In [ ]:
"""
Cell F3 — Per-Trip Aggregations (PySpark)

Purpose
-------
Aggregate second-level sensor features into trip-level features (one row per booking_id).

For each booking_id, compute:
    - FE_n_samples           : number of sensor rows in the trip
    - FE_trip_duration_s     : max(second) - min(second)
    - FE_sampling_rate_hz    : FE_n_samples / FE_trip_duration_s

Speed behaviour:
    - FE_speed_mean, FE_speed_std, FE_speed_max, FE_speed_min
    - FE_speed_p90           : 90th percentile of speed
    - FE_speed_change_rate_med : median(|Δv| / Δt)

Acceleration magnitude:
    - FE_acc_mag_mean, FE_acc_mag_std, FE_acc_mag_max, FE_acc_mag_p95

Axis-level acceleration:
    - FE_acc_x_mean, FE_acc_y_mean, FE_acc_z_mean
    - FE_acc_x_std,  FE_acc_y_std,  FE_acc_z_std

Jerk:
    - FE_jerk_med  : median jerk
    - FE_jerk_p95  : 95th percentile of jerk

Gyro magnitude:
    - FE_gyro_mag_mean, FE_gyro_mag_std, FE_gyro_mag_p95

Turning:
    - FE_turn_delta_std_deg : std-dev of bearing_diff_deg
    - FE_turn_rate_med_deg_s: median turn_rate_deg_s

GPS accuracy:
    - FE_gps_acc_mean, FE_gps_acc_std
"""


# Core per-trip aggregates (1 row per booking_id)
trip_aggs = dfc.groupBy("booking_id").agg(
    # Meta
    F.count(F.lit(1)).alias("FE_n_samples"),
    (F.max("second") - F.min("second")).alias("FE_trip_duration_s"),

    # Speed behaviour
    F.avg("speed").alias("FE_speed_mean"),
    F.stddev_pop("speed").alias("FE_speed_std"),
    F.max("speed").alias("FE_speed_max"),
    F.min("speed").alias("FE_speed_min"),
    F.expr("percentile_approx(speed, 0.90)").alias("FE_speed_p90"),
    F.expr("percentile_approx(speed_change_rate, 0.50)").alias("FE_speed_change_rate_med"),

    # Acceleration magnitude
    F.avg("acc_magnitude").alias("FE_acc_mag_mean"),
    F.stddev_pop("acc_magnitude").alias("FE_acc_mag_std"),
    F.max("acc_magnitude").alias("FE_acc_mag_max"),
    F.expr("percentile_approx(acc_magnitude, 0.95)").alias("FE_acc_mag_p95"),

    # Axis-level acceleration
    F.avg("acceleration_x").alias("FE_acc_x_mean"),
    F.avg("acceleration_y").alias("FE_acc_y_mean"),
    F.avg("acceleration_z").alias("FE_acc_z_mean"),
    F.stddev_pop("acceleration_x").alias("FE_acc_x_std"),
    F.stddev_pop("acceleration_y").alias("FE_acc_y_std"),
    F.stddev_pop("acceleration_z").alias("FE_acc_z_std"),

    # Jerk
    F.expr("percentile_approx(jerk, 0.50)").alias("FE_jerk_med"),
    F.expr("percentile_approx(jerk, 0.95)").alias("FE_jerk_p95"),

    # Gyro magnitude
    F.avg("gyro_magnitude").alias("FE_gyro_mag_mean"),
    F.stddev_pop("gyro_magnitude").alias("FE_gyro_mag_std"),
    F.expr("percentile_approx(gyro_magnitude, 0.95)").alias("FE_gyro_mag_p95"),

    # Turning
    F.stddev_pop("bearing_diff_deg").alias("FE_turn_delta_std_deg"),
    F.expr("percentile_approx(turn_rate_deg_s, 0.50)").alias("FE_turn_rate_med_deg_s"),

    # GPS accuracy
    F.avg("accuracy").alias("FE_gps_acc_mean"),
    F.stddev_pop("accuracy").alias("FE_gps_acc_std"),
)

# Sampling rate (Hz) = n_samples / duration (only if duration > 0)
trip_features = trip_aggs.withColumn(
    "FE_sampling_rate_hz",
    F.when(
        F.col("FE_trip_duration_s") > 0,
        F.col("FE_n_samples") / F.col("FE_trip_duration_s")
    )
)

print("Cell F3 complete — core FE_ features computed per booking_id.")
trip_features.select(
    "booking_id", "FE_n_samples", "FE_trip_duration_s",
    "FE_sampling_rate_hz", "FE_speed_mean", "FE_acc_mag_mean"
).show(10, truncate=False)

### Cell 5 — Aggregate to per-trip table (with start/end speeds)

**Purpose.** Build the final **engineered feature dataset** by:

1. Extracting `FE_start_speed` and `FE_end_speed` per `booking_id` using window functions.
2. Joining these edge features into the aggregated `trip_features` table.

**What this produces**
- One row per trip with ~30–35 `FE_` features capturing speed discipline, motion abruptness, turning stability, and signal quality.
- This compact table is ideal for:
  - **EDA**: boxplots/histograms of `FE_jerk_p95`, `FE_turn_rate_med_deg_s`, etc.
  - **Dashboarding**: rank trips/drivers by `FE_speed_p90`, flag high-risk profiles.
  - **Modeling (Sprint 2)**: feed into classifiers/regressors to detect **dangerous driving**.

**Post-processing**
- (Optional) Drop temporary row-level columns (`acc_magnitude`, `gyro_magnitude`, `delta_*`, etc.) from `dfc` if you don’t need them further.
- Keep `booking_id` as the key for merging with drivers/safety labels.


In [ ]:
"""
Cell F4 — Start/End Speed & Final Trip Feature Table (PySpark)

Purpose
-------
For each booking_id (trip), add:
    - FE_start_speed : speed at earliest timestamp (second)
    - FE_end_speed   : speed at latest timestamp (second)

Then merge these into `trip_features` to produce a final
per-trip feature table ready for downstream ML / DW loading.
"""


# Windows for first and last sample per trip
w_first = Window.partitionBy("booking_id").orderBy("second")
w_last  = Window.partitionBy("booking_id").orderBy(F.col("second").desc())

# Attach row numbers for first/last detection
df_start_end = (
    dfc
    .withColumn("rn_first", F.row_number().over(w_first))
    .withColumn("rn_last",  F.row_number().over(w_last))
)

# Start speed per trip (earliest second)
start_speed_df = (
    df_start_end
    .filter(F.col("rn_first") == 1)
    .select("booking_id", F.col("speed").alias("FE_start_speed"))
)

# End speed per trip (latest second)
end_speed_df = (
    df_start_end
    .filter(F.col("rn_last") == 1)
    .select("booking_id", F.col("speed").alias("FE_end_speed"))
)

# Join onto existing trip-level features
trip_features = (
    trip_features
    .join(start_speed_df, on="booking_id", how="left")
    .join(end_speed_df,  on="booking_id", how="left")
)

print("Cell F4 complete — final trip-level feature table ready.")
trip_features.show(10, truncate=False)

### 3.6.6 How These Features Help Detect Dangerous Driving

All engineered features correspond to real-world driving physics and human risk behaviors:

| Category | Driving Behavior Captured | Dangerous Pattern Detected |
|-----------|---------------------------|-----------------------------|
| **Speed Metrics** | Average, peak, and variability of velocity | Speeding and harsh braking |
| **Acceleration Metrics** | Force and frequency of acceleration | Rapid bursts and jerky motion |
| **Jerk Metrics** | Change in acceleration over time | Sudden braking or acceleration |
| **Turning Metrics** | Bearing and gyro variability | Swerving or cornering too fast |
| **Axis-Level Variance** | Directional motion instability | Lane weaving, uneven control |
| **Gyro Metrics** | Rotational control | Oversteering, instability |
| **Trip Context** | Duration, sampling, GPS accuracy | Distinguish genuine vs. noisy trips |
| **Time-of-Day** | Driving conditions (day/night) | Riskier night driving patterns |

Together, these features quantify **how controlled or erratic** a trip is.  
A trip with high jerk, turning volatility, and speed variability signals **dangerous driving**,  
while a stable profile with low acceleration variance and smooth turns indicates **safe behavior**.

---
## 3.7 Data Audit

In this sub-section, we will be auditing the Data ETL Pipeline by verifying if all the Data is deemed cleaned and transformed. Should the Data pass the audit, we will proceed with the Load process of the ETL pipeline where we push the changes to the Data Warehouse in MySQL Database.

In [ ]:
"""
Cell C9 — AFTER Audit & Optional Persist (Spark Version)

Purpose
-------
Compute AFTER-cleaning audit metrics for the final cleaned Spark
DataFrame (df_clean), including:
    • Shape (n_rows, n_cols)
    • Missing values per column
    • Dtype snapshot
    • Duplicate count (full-row)

Optionally: persist df_clean to Parquet/CSV/MySQL DW.

Notes
-----
- df_clean is the final cleaned Spark DataFrame
  (output from C8 after MV outlier scoring/filtering).
- All heavy computations are Spark-native.
- Pandas is only used for small display tables.
"""


# ---------------------------------------------------------------------
# 0. Sanity check
# ---------------------------------------------------------------------
print("[C9] Running AFTER-audit on df_clean")
print("Columns in df_clean:", df_clean.columns)

# ---------------------------------------------------------------------
# 1. Shape
# ---------------------------------------------------------------------
after_rows = df_clean.count()
after_cols = len(df_clean.columns)

print(f"[AFTER] Shape = ({after_rows:,}, {after_cols})")


# ---------------------------------------------------------------------
# 2. Missing values per column (Spark → small pandas for display)
# ---------------------------------------------------------------------
missing_df = df_clean.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in df_clean.columns
])

missing_pdf = (
    missing_df.toPandas()
    .T
    .reset_index()
    .rename(columns={"index": "Column", 0: "Missing Count"})
)
missing_pdf["Missing %"] = (
    missing_pdf["Missing Count"] / after_rows * 100
).round(2)

print("\n[AFTER] Missing value summary:")
display(missing_pdf.head(12))


# ---------------------------------------------------------------------
# 3. Dtype snapshot
# ---------------------------------------------------------------------
dtype_pdf = pd.DataFrame(
    df_clean.dtypes, columns=["Column", "Spark DataType"]
)

print("\n[AFTER] Dtype snapshot:")
display(dtype_pdf)


# ---------------------------------------------------------------------
# 4. Duplicate count (full-row equality)
# ---------------------------------------------------------------------
after_dups = after_rows - df_clean.dropDuplicates().count()

dup_summary_pdf = pd.DataFrame([{
    "Total Rows": after_rows,
    "Duplicate Count": after_dups,
    "Duplicate %": round(after_dups / after_rows * 100, 5) if after_rows else 0,
    "Status": "Duplicates Found" if after_dups > 0 else "No Duplicates",
}])

print("\n[AFTER] Duplicate summary:")
display(dup_summary_pdf)


# ---------------------------------------------------------------------
# 5. (Optional) Persist Cleaned Data — choose one
# ---------------------------------------------------------------------
# Options:
#   None      → no write
#   "parquet" → write to Parquet folder
#   "csv"     → write to CSV folder
#   "mysql"   → write to MySQL DW using JDBC (requires MySQL JAR on Spark classpath)
# ---------------------------------------------------------------------

SAVE_TO = None      # "parquet", "csv", or "mysql"
SAVE_PATH = "/content/df_clean"   # local path (Colab). Adjust if needed.

if SAVE_TO == "parquet":
    (df_clean.write
        .mode("overwrite")
        .parquet(SAVE_PATH))
    print(f"\n[WRITE] Saved cleaned dataset to Parquet → {SAVE_PATH}")

elif SAVE_TO == "csv":
    (df_clean.write
        .mode("overwrite")
        .option("header", True)
        .csv(SAVE_PATH))
    print(f"\n[WRITE] Saved cleaned dataset to CSV folder → {SAVE_PATH}")

elif SAVE_TO == "mysql":
    # --- DW / warehouse config (same RDS host, different schema) ---
    dw_host = os.getenv(
        "DW_DB_HOST",
        "your-db-host.example.com"
    )
    dw_port = int(os.getenv("DW_DB_PORT", "3306"))
    dw_name = os.getenv("DW_DB_NAME", "warehouse")   # your DW schema
    dw_user = os.getenv("DW_DB_USER", "admin")
    dw_pass = os.getenv("DW_DB_PASSWORD")

    # Final fact/staging table name in DW
    dw_table = os.getenv("DW_DB_TABLE", "Fact_Cleaned_Sensor")

    jdbc_url = (
        f"jdbc:mysql://{dw_host}:{dw_port}/{dw_name}"
        "?serverTimezone=UTC&characterEncoding=UTF-8"
    )

    print("\n[WRITE] Writing df_clean to MySQL DW ...")
    print("  URL   :", jdbc_url)
    print("  Table :", dw_table)

    (df_clean.write
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", dw_table)
        .option("user", dw_user)
        .option("password", dw_pass)
        .option("driver", "com.mysql.cj.jdbc.Driver")
        .mode("append")
        .save())

    print("[WRITE] df_clean successfully appended to MySQL DW.")

else:
    print("\n[WRITE] SAVE_TO is None — no persistence performed in C9.")


With reference to the Code Cell above, we are able to determine that the Sensor Data have been cleaned successfully and we are ready to proceed to conduct the Load Layer of thge ETL pipeline.

---
# 4. Data Load Layer

In this section, we will be loading the Data in to the Data Warehouse in the MySQL Database. This process is known as the Load process of the ETL Pipeline. We will have the ensure that the Data is competiable with the MySQL syntax and can be Loaded successfully. We will also be adding a surropgate key as per the convention used for Data Warehouses so as to maintain integrity rules. The Data Loading operation will be conducted in the Code Cell below.

In [ ]:
"""
Cell C11 — Load trip_features into Dim_Sensor (streaming) + Mark Staging

Updated version (fix):
- Uses df_clean as the source for raw sensor aggregates instead of df_typed.
- Ensures the final DataFrame has BOTH:
    • Aggregated raw columns: accuracy, bearing, accel_*, gyro_*, second, speed
    • Feature-engineered columns: FE_*
- Aligns the output to the Dim_Sensor schema and loads to MySQL (warehouse.Dim_Sensor).
- Marks the corresponding staging.sensor_data_source rows as cleaned.

Grain
-----
1 row per booking_id (trip-level profile).

Inputs (globals)
----------------
df_clean : Spark DataFrame
    Output of C8 — fully cleaned row-level sensor data
    (includes original columns + outlier flags, etc.).

trip_features : Spark DataFrame
    Output of F3 — per-trip aggregation of FE_* features.

db_user, db_pass, db_host, db_port, db_name, db_table
    Connection info for the STAGING database (sensor_data_source table).

Environment Variables (optional overrides)
------------------------------------------
DW_USER, DW_PASS, DW_HOST, DW_PORT, DW_DB, DW_TABLE
    If set, override the default DW connection and table.
"""


# ---------------------------------------------------------------------
# 0. Basic checks & DW config
# ---------------------------------------------------------------------
assert "trip_features" in globals(), "[C11] trip_features not found. Run F3 first."
assert "df_clean" in globals(), "[C11] df_clean not found. Run C8 first."

if "booking_id" not in trip_features.columns:
    raise ValueError(
        "[C11] ERROR: 'booking_id' column is missing from trip_features. "
        "Cannot populate Dim_Sensor without the natural key."
    )

# Data Warehouse (Dim_Sensor) connection (warehouse)
DW_USER = os.getenv("DW_USER", db_user)
DW_PASS = os.getenv("DW_PASS", db_pass)
DW_HOST = os.getenv("DW_HOST", db_host)
DW_PORT = int(os.getenv("DW_PORT", db_port))
DW_DB = os.getenv("DW_DB", "warehouse")
DW_TABLE = os.getenv("DW_TABLE", "Dim_Sensor")  # must match your DW DDL

print(f"[C11] DW target: {DW_HOST}:{DW_PORT}/{DW_DB}.{DW_TABLE}")

# ---------------------------------------------------------------------
# 1. Aggregate RAW sensor values per booking_id (from df_clean)
# ---------------------------------------------------------------------
df_source = df_clean  # <- use fully cleaned data, not df_typed

raw_cols_required = [
    "booking_id",
    "accuracy",
    "bearing",
    "acceleration_x",
    "acceleration_y",
    "acceleration_z",
    "gyro_x",
    "gyro_y",
    "gyro_z",
    "second",
    "speed",
]

missing_raw = [c for c in raw_cols_required if c not in df_source.columns]
if missing_raw:
    raise ValueError(
        "[C11] df_clean is missing required raw columns for Dim_Sensor aggregation: "
        f"{missing_raw}"
    )

raw_base = (
    df_source
    .select(raw_cols_required)
    .withColumn("booking_id", F.col("booking_id").cast("bigint"))
    .withColumn("accuracy", F.col("accuracy").cast("double"))
    .withColumn("bearing", F.col("bearing").cast("double"))
    .withColumn("acceleration_x", F.col("acceleration_x").cast("double"))
    .withColumn("acceleration_y", F.col("acceleration_y").cast("double"))
    .withColumn("acceleration_z", F.col("acceleration_z").cast("double"))
    .withColumn("gyro_x", F.col("gyro_x").cast("double"))
    .withColumn("gyro_y", F.col("gyro_y").cast("double"))
    .withColumn("gyro_z", F.col("gyro_z").cast("double"))
    .withColumn("second", F.col("second").cast("int"))
    .withColumn("speed", F.col("speed").cast("double"))
)

raw_agg = (
    raw_base
    .groupBy("booking_id")
    .agg(
        F.avg("accuracy").alias("accuracy"),
        F.avg("bearing").alias("bearing"),
        F.avg("acceleration_x").alias("acceleration_x"),
        F.avg("acceleration_y").alias("acceleration_y"),
        F.avg("acceleration_z").alias("acceleration_z"),
        F.avg("gyro_x").alias("gyro_x"),
        F.avg("gyro_y").alias("gyro_y"),
        F.avg("gyro_z").alias("gyro_z"),
        F.max("second").cast("int").alias("second"),  # last second in trip
        F.avg("speed").alias("speed"),                # average speed over trip
    )
)

print("[C11] raw_agg columns:", raw_agg.columns)

# ---------------------------------------------------------------------
# 2. Join FE_* trip_features with raw_agg on booking_id
# ---------------------------------------------------------------------
trip_base = trip_features.withColumn(
    "booking_id", F.col("booking_id").cast("bigint")
)

trip_export = (
    trip_base.alias("fe")
    .join(raw_agg.alias("raw"), on="booking_id", how="left")
)

print("[C11] trip_export columns after join:", trip_export.columns)

# ---------------------------------------------------------------------
# 3. Cast numerics & add ETL timestamp
# ---------------------------------------------------------------------
numeric_cols = [
    "accuracy",
    "bearing",
    "acceleration_x",
    "acceleration_y",
    "acceleration_z",
    "gyro_x",
    "gyro_y",
    "gyro_z",
    "second",
    "speed",
    "FE_n_samples",
    "FE_trip_duration_s",
    "FE_sampling_rate_hz",
    "FE_speed_mean",
    "FE_speed_std",
    "FE_speed_max",
    "FE_speed_min",
    "FE_speed_p90",
    "FE_speed_change_rate_med",
    "FE_acc_mag_mean",
    "FE_acc_mag_std",
    "FE_acc_mag_max",
    "FE_acc_mag_p95",
    "FE_acc_x_mean",
    "FE_acc_y_mean",
    "FE_acc_z_mean",
    "FE_acc_x_std",
    "FE_acc_y_std",
    "FE_acc_z_std",
    "FE_jerk_med",
    "FE_jerk_p95",
    "FE_gyro_mag_mean",
    "FE_gyro_mag_std",
    "FE_gyro_mag_p95",
    "FE_turn_delta_std_deg",
    "FE_turn_rate_med_deg_s",
    "FE_gps_acc_mean",
    "FE_gps_acc_std",
    "FE_start_speed",
    "FE_end_speed",
]

for col_name in numeric_cols:
    if col_name in trip_export.columns:
        if col_name in ["second", "FE_n_samples", "FE_trip_duration_s"]:
            trip_export = trip_export.withColumn(col_name, F.col(col_name).cast("int"))
        else:
            trip_export = trip_export.withColumn(col_name, F.col(col_name).cast("double"))

trip_export = trip_export.withColumn("date_cleaned", F.current_timestamp())

# ---------------------------------------------------------------------
# 4. Align columns to Dim_Sensor schema (original + FE_* + date_cleaned)
# ---------------------------------------------------------------------
dim_sensor_cols = [
    "booking_id",
    "accuracy",
    "bearing",
    "acceleration_x",
    "acceleration_y",
    "acceleration_z",
    "gyro_x",
    "gyro_y",
    "gyro_z",
    "second",
    "speed",
    "FE_n_samples",
    "FE_trip_duration_s",
    "FE_sampling_rate_hz",
    "FE_speed_mean",
    "FE_speed_std",
    "FE_speed_max",
    "FE_speed_min",
    "FE_speed_p90",
    "FE_speed_change_rate_med",
    "FE_acc_mag_mean",
    "FE_acc_mag_std",
    "FE_acc_mag_max",
    "FE_acc_mag_p95",
    "FE_acc_x_mean",
    "FE_acc_y_mean",
    "FE_acc_z_mean",
    "FE_acc_x_std",
    "FE_acc_y_std",
    "FE_acc_z_std",
    "FE_jerk_med",
    "FE_jerk_p95",
    "FE_gyro_mag_mean",
    "FE_gyro_mag_std",
    "FE_gyro_mag_p95",
    "FE_turn_delta_std_deg",
    "FE_turn_rate_med_deg_s",
    "FE_gps_acc_mean",
    "FE_gps_acc_std",
    "FE_start_speed",
    "FE_end_speed",
    "date_cleaned",
]

# Ensure all expected columns exist (add NULLs if missing, just in case)
for col_name in dim_sensor_cols:
    if col_name not in trip_export.columns:
        if col_name in ["booking_id", "FE_n_samples", "FE_trip_duration_s", "second"]:
            trip_export = trip_export.withColumn(col_name, F.lit(None).cast("int"))
        elif col_name == "date_cleaned":
            trip_export = trip_export.withColumn(col_name, F.current_timestamp())
        else:
            trip_export = trip_export.withColumn(col_name, F.lit(None).cast("double"))

trip_export = trip_export.select(dim_sensor_cols)
dw_cols = dim_sensor_cols
print("[C11] DW column order:", dw_cols)

# ---------------------------------------------------------------------
# 5. Build INSERT statement
# ---------------------------------------------------------------------
placeholders = ", ".join(["%s"] * len(dw_cols))
col_list = ", ".join(f"`{c}`" for c in dw_cols)

insert_sql = f"""
    INSERT INTO {DW_DB}.{DW_TABLE} ({col_list})
    VALUES ({placeholders})
"""

# ---------------------------------------------------------------------
# 6. Connect to DW and stream-insert in batches
# ---------------------------------------------------------------------
dw_conn = mysql.connector.connect(
    host=DW_HOST,
    port=DW_PORT,
    user=DW_USER,
    password=DW_PASS,
    database=DW_DB,
)
dw_cur = dw_conn.cursor()

BATCH_SIZE = 1000
batch = []
total_inserted = 0
booking_ids_set = set()

print("[C11] Starting streaming insert from Spark → DW.Dim_Sensor ...")

for row in trip_export.toLocalIterator():
    row_tuple = tuple(row[c] for c in dw_cols)
    batch.append(row_tuple)
    booking_ids_set.add(row["booking_id"])

    if len(batch) >= BATCH_SIZE:
        dw_cur.executemany(insert_sql, batch)
        dw_conn.commit()
        total_inserted += len(batch)
        print(f"[C11] Inserted batch, cumulative rows: {total_inserted}")
        batch = []

# Flush remaining rows
if batch:
    dw_cur.executemany(insert_sql, batch)
    dw_conn.commit()
    total_inserted += len(batch)
    print(f"[C11] Final batch inserted. Total rows inserted/updated: {total_inserted}")

dw_cur.close()
dw_conn.close()
print("[C11] DW Dim_Sensor load complete.")

# ---------------------------------------------------------------------
# 7. Mark staging.sensor_data_source rows as cleaned
# ---------------------------------------------------------------------
booking_ids = [bid for bid in booking_ids_set if bid is not None]
print(f"[C11] Unique booking_ids for staging update: {len(booking_ids)}")

if booking_ids:
    staging_conn = mysql.connector.connect(
        host=db_host,
        port=db_port,
        user=db_user,
        password=db_pass,
        database=db_name,  # e.g. "staging"
    )
    staging_cur = staging_conn.cursor()

    chunk_size = 1000
    total_updated = 0

    for i in range(0, len(booking_ids), chunk_size):
        chunk = booking_ids[i: i + chunk_size]
        placeholders = ", ".join(["%s"] * len(chunk))

        sql = f"""
            UPDATE {db_table}   -- sensor_data_source
            SET cleaned_date = NOW()
            WHERE booking_id IN ({placeholders})
        """

        staging_cur.execute(sql, chunk)
        staging_conn.commit()
        total_updated += staging_cur.rowcount
        print(f"[C11] Staging UPDATE chunk done, cumulative updated: {total_updated}")

    staging_cur.close()
    staging_conn.close()
    print(
        f"[C11] Staging update complete — cleaned_date set for ~{total_updated} rows "
        f"in '{db_table}'."
    )
else:
    print("[C11] No booking_ids collected — staging.cleaned_date not updated.")

With reference to the Code Cell above, we are able to determine that the Data Loading have been completed and we have successfully completed the ETL Pipeline.